# 3. Gün: Çok Değişkenli Oynaklık Modelleri
## DCC-GARCH, cDCC, ADCC, DECO & Minimum Varyans Portföy
### EYS'26 — Pamukkale Üniversitesi

Bu not defteri, ders uygulamasında kullanılan hesaplamaları adım adım açıklar.
Veri: `../data/sample_returns.csv`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

# Ders modülleri (python/ klasöründe)
import sys, os
sys.path.insert(0, str(Path('..').resolve()))

from dcc_garch import DCCGarch

plt.style.use('dark_background')
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})
print("Modüller yüklendi.")

In [ ]:
DATA_PATH = Path('..') / 'data' / 'sample_returns.csv'
df = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True)

# Sadece getiri sütunları (RV ve BPV hariç)
asset_cols = [c for c in df.columns if not c.endswith('_RV') and not c.endswith('_BPV')]
returns = df[asset_cols]

print(f"Varlık sayısı : {len(asset_cols)}")
print(f"Gözlem sayısı : {len(returns)}")
print(f"Tarih aralığı : {returns.index[0].date()} — {returns.index[-1].date()}")
print(f"\nVarlıklar: {asset_cols}")
returns.describe().round(6)

## 1. DCC-GARCH Modeli (Engle 2002)

**İki aşamalı tahmin:**
1. Her varlık için GARCH(1,1) → koşullu standart sapmalar $\sigma_{it}$
2. Standardize artıklar $z_{it} = r_{it}/\sigma_{it}$ ile DCC parametreleri $(a, b)$ tahmin edilir.

**Korelasyon dinamiği:**
$$Q_t = (1-a-b)\bar{Q} + a\, z_{t-1}z_{t-1}^\top + b\, Q_{t-1}$$
$$R_t = Q_t^{*-1} Q_t Q_t^{*-1}$$

In [ ]:
# İlk 3 varlık ile DCC tahmini
sel = asset_cols[:3]
model_dcc = DCCGarch(returns[sel].values, model='DCC')
res_dcc = model_dcc.fit()

print("=== DCC Parametre Tahmini ===")
print(f"  alpha (a) : {res_dcc['stats']['alpha']:.6f}")
print(f"  beta  (b) : {res_dcc['stats']['beta']:.6f}")
print(f"  a + b     : {res_dcc['stats']['persistence']:.6f}")
print(f"  Yarı-ömür : {res_dcc['stats']['half_life_days']:.1f} gün")
print(f"  Ort. kor. : {res_dcc['stats']['mean_corr']:.4f}")

In [ ]:
corr_series = res_dcc['corr_series']
pair = f"{sel[0]} vs {sel[1]}"
rho = corr_series[pair]
idx = res_dcc['index']

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(idx, rho, lw=1.5, color='#56B4E9', label=pair)
ax.axhline(np.mean(rho), ls='--', color='#E69F00', lw=1, label=f'Ort. ρ = {np.mean(rho):.3f}')
ax.set_title(f'Dinamik Koşullu Korelasyon — DCC — {pair}')
ax.set_xlabel('Tarih'); ax.set_ylabel('ρ_t')
ax.legend(); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout(); plt.show()

## 2. cDCC (Aielli 2013) ve ADCC (Cappiello vd. 2006)

**cDCC:** $\tilde{z}_{t-1} = Q_{t-1}^{*1/2} z_{t-1}$ ile $\bar{Q}$'yu tutarlı tahmin eder.

**ADCC:** Negatif şoklar $n_t = z_t \odot \mathbf{1}[z_t<0]$ ile asimetri terimi ekler.
$$Q_t = (1-a-b)\bar{Q} - c\bar{N} + a\,z_{t-1}z_{t-1}^\top + b\,Q_{t-1} + c\,n_{t-1}n_{t-1}^\top$$

In [ ]:
results_cmp = {}
for mtype in ['DCC', 'cDCC', 'ADCC']:
    m = DCCGarch(returns[sel].values, model=mtype)
    results_cmp[mtype] = m.fit()
    s = results_cmp[mtype]['stats']
    print(f"{mtype:6s}  a={s['alpha']:.4f}  b={s['beta']:.4f}  "
          f"a+b={s['persistence']:.4f}  HL={s['half_life_days']:.1f}g  "
          f"ρ̄={s['mean_corr']:.4f}")

In [ ]:
R_dcc  = np.asarray(results_cmp['DCC']['R_seq'])
R_adcc = np.asarray(results_cmp['ADCC']['R_seq'])
idx_c  = results_cmp['DCC']['index']
iu = np.triu_indices(len(sel), k=1)

mean_dcc  = R_dcc[:,  iu[0], iu[1]].mean(1)
mean_adcc = R_adcc[:, iu[0], iu[1]].mean(1)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(idx_c, mean_dcc,  lw=1.4, color='#999999', label='DCC')
ax1.plot(idx_c, mean_adcc, lw=1.4, color='#009E73', label='ADCC')
ax1.set_title('Ortalama Koşullu Korelasyon: DCC vs ADCC')
ax1.set_ylabel('Ort. ρ_t'); ax1.legend()

ax2.plot(idx_c, mean_adcc - mean_dcc, lw=1.0, color='#009E73')
ax2.axhline(0, color='gray', lw=0.8, ls='--')
ax2.fill_between(idx_c, mean_adcc - mean_dcc, 0, alpha=0.3, color='#009E73')
ax2.set_title('Fark: ADCC − DCC (c > 0 → kriz döneminde daha yüksek kor.)')
ax2.set_ylabel('ADCC − DCC')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout(); plt.show()

## 3. DECO — Dinamik Ekikorelasyon (Engle-Kelly 2012)

Tüm çiftlerin ortalaması tek bir skaler $\rho_t$ ile modellenir:
$$\rho_t = \frac{2}{N(N-1)}\sum_{i<j} R_{ij,t}^{\text{DCC}}$$

$N > 50$'de DCC'ye göre $O(1)$ parametre avantajı sağlar.

In [ ]:
# Tüm varlıklar ile DECO
model_deco = DCCGarch(returns[asset_cols].values, model='DECO')
res_deco = model_deco.fit()

rho_deco = res_deco['rho_series']
idx_deco = res_deco['index']

print(f"DECO alpha : {res_deco['stats']['alpha']:.4f}")
print(f"DECO beta  : {res_deco['stats']['beta']:.4f}")
print(f"Ort. ρ_t   : {res_deco['stats']['mean_corr']:.4f}")

fig, ax = plt.subplots(figsize=(12, 4))
roll = pd.Series(rho_deco, index=idx_deco).rolling(60, min_periods=1).mean()
ax.fill_between(idx_deco, rho_deco, alpha=0.25, color='#a78bfa')
ax.plot(idx_deco, rho_deco, lw=0.8, color='#a78bfa', label='DECO ρ_t')
ax.plot(roll.index, roll.values, lw=2, color='#E69F00', ls='--', label='60g kayan ort.')
ax.axhline(res_deco['stats']['sample_mean_corr'], ls=':', color='#56B4E9',
           label=f"Örnek ort. = {res_deco['stats']['sample_mean_corr']:.3f}")
ax.set_title('DECO Ekikorelasyon Serisi'); ax.set_ylabel('ρ_t'); ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout(); plt.show()

## 4. Minimum Varyans Portföy (MVP)

$$w_t = \frac{H_t^{-1}\mathbf{1}}{\mathbf{1}^\top H_t^{-1}\mathbf{1}}$$

Her $t$ için DCC koşullu kovaryans $H_t = D_t R_t D_t$ kullanılarak dinamik MVP ağırlıkları hesaplanır.

In [ ]:
# DCC ile MVP
model_mvp = DCCGarch(returns[sel].values, model='DCC')
res_mvp = model_mvp.fit()
H_seq = np.asarray(res_mvp['H_seq'])  # (T, N, N)
idx_m = res_mvp['index']
N = len(sel); T = H_seq.shape[0]
ones = np.ones(N)

W = np.zeros((T, N))
for t in range(T):
    H_inv = np.linalg.pinv(H_seq[t])
    denom = ones @ H_inv @ ones
    W[t] = (H_inv @ ones) / denom if denom > 1e-12 else np.full(N, 1/N)

W_df = pd.DataFrame(W, index=idx_m, columns=sel)
ann = np.sqrt(252)
pv = np.array([np.sqrt(W[t] @ H_seq[t] @ W[t]) for t in range(T)])

print(f"Ort. portföy oynaklığı (günlük): {pv.mean()*100:.4f}%")
print(f"Yıllık oynaklık tahmini        : {pv.mean()*100*ann:.2f}%")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for i, c in enumerate(sel):
    ax1.plot(idx_m, W[:, i], lw=1.2, label=c)
ax1.set_title('Dinamik MVP Ağırlıkları (DCC)'); ax1.set_ylabel('w_t'); ax1.legend()
ax2.plot(idx_m, pv * 100, lw=1.5, color='#f472b6')
ax2.set_title('MVP Portföy Oynaklığı (günlük %)')
ax2.set_ylabel('σ_p (%)'); ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout(); plt.show()

## Özet

| Model | Parametre | Avantaj | Ne Zaman? |
|-------|-----------|---------|----------|
| DCC   | a, b      | Hızlı, yorumlanabilir | N < 20 |
| cDCC  | a, b      | Tutarlı Q̄ tahmini | Yüksek kalıcılık |
| ADCC  | a, b, c   | Asimetri (c > 0) | Kriz dönemi |
| DECO  | a, b      | O(1) parametre | N > 50 |

**Kaynaklar:**
- Engle, R. F. (2002). Dynamic Conditional Correlation. *JBES*, 20(3), 339–350.
- Aielli, G. P. (2013). Dynamic Conditional Correlation. *JBES*, 31(4), 482–491.
- Cappiello, L., Engle, R. F., & Sheppard, K. (2006). Asymmetric Dynamics. *JFEC*, 4(4), 537–572.
- Engle, R., & Kelly, B. (2012). Dynamic Equicorrelation. *JBES*, 30(2), 212–228.